In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

In [8]:
# ============================================================
# 1. LOAD DATA
# ============================================================
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train.head()


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [10]:
test.head()

,id,prompt,response_a,response_b
0,136060,"[""I have three oranges today, I ate an orange ...","[""You have two oranges today.""]","[""You still have three oranges. Eating an oran..."
1,211333,"[""You are a mediator in a heated political deb...","[""Thank you for sharing the details of the sit...","[""Mr Reddy and Ms Blue both have valid points ..."
2,1233961,"[""How to initialize the classification head wh...","[""When you want to initialize the classificati...","[""To initialize the classification head when p..."


Data cleaning the csv files

In [11]:
# ============================================================
# 2. DATA CLEANING
# ============================================================
for col in ['prompt', 'response_a', 'response_b']:
    train[col] = train[col].fillna('')
    test[col] = test[col].fillna('')

train = train.drop_duplicates().reset_index(drop=True)
print(f"Train rows: {len(train)}, Test rows: {len(test)}")

Train rows: 57477, Test rows: 3


For the model features, we'll start by generating a bunch of easy ones based on text characteristics from the models and prompts, like word and character count. 

In [13]:
# ============================================================
# 3. BUILD FEATURES
# ============================================================
def build_features(df):
    f = pd.DataFrame()
    # Length features
    f['len_a'] = df['response_a'].str.len()
    f['len_b'] = df['response_b'].str.len()
    f['len_diff'] = f['len_a'] - f['len_b']
    f['len_ratio'] = f['len_a'] / (f['len_b'] + 1)
    f['len_prompt'] = df['prompt'].str.len()
    # Word count features
    f['words_a'] = df['response_a'].str.split().str.len()
    f['words_b'] = df['response_b'].str.split().str.len()
    f['words_diff'] = f['words_a'] - f['words_b']
    # Structural features
    f['newlines_a'] = df['response_a'].str.count('\n')
    f['newlines_b'] = df['response_b'].str.count('\n')
    f['code_a'] = df['response_a'].str.count('```')
    f['code_b'] = df['response_b'].str.count('```')
    f['bullets_a'] = df['response_a'].str.count(r'\n\s*[-*]')
    f['bullets_b'] = df['response_b'].str.count(r'\n\s*[-*]')
    return f

train_features = build_features(train)
test_features = build_features(test)

print(test_features.head())

   len_a  len_b  len_diff  len_ratio  len_prompt  words_a  words_b  \
0     31    114       -83   0.269565          86        5       19   
1   1457    460       997   3.160521         488      217       75   
2   3984   3716       268   1.071832         217      623      437   

   words_diff  newlines_a  newlines_b  code_a  code_b  bullets_a  bullets_b  
0         -14           0           0       0       0          0          0  
1         142           0           0       0       0          0          0  
2         186           0           0       0       4          0          0  


Using TF-IDF, we can determine what topics the models go over, and if specific topics correlate to a better model response.

In [14]:
# ============================================================
# 4. TF-IDF FEATURES
# ============================================================
# Prompt TF-IDF — captures the topic
tfidf_prompt = TfidfVectorizer(max_features=200)
tfidf_prompt.fit(train['prompt'])
train_tfidf_prompt = tfidf_prompt.transform(train['prompt']).toarray()
test_tfidf_prompt = tfidf_prompt.transform(test['prompt']).toarray()

# Response TF-IDF — fit on both responses so they share a vocabulary
tfidf_resp = TfidfVectorizer(max_features=200)
tfidf_resp.fit(pd.concat([train['response_a'], train['response_b']]))

train_tfidf_a = tfidf_resp.transform(train['response_a']).toarray()
train_tfidf_b = tfidf_resp.transform(train['response_b']).toarray()
test_tfidf_a = tfidf_resp.transform(test['response_a']).toarray()
test_tfidf_b = tfidf_resp.transform(test['response_b']).toarray()

# Difference highlights where the two responses diverge
train_tfidf_diff = train_tfidf_a - train_tfidf_b
test_tfidf_diff = test_tfidf_a - test_tfidf_b

# Combine all features
X_all = np.hstack([train_features.values, train_tfidf_prompt, train_tfidf_diff])
X_test = np.hstack([test_features.values, test_tfidf_prompt, test_tfidf_diff])
print(f"Total features: {X_all.shape[1]}")

Total features: 414


Target prediction (model a, model b, tie)

In [19]:
# ============================================================
# 5. PREPARE TARGET
# ============================================================
y = train[['winner_model_a', 'winner_model_b', 'winner_tie']].values.argmax(axis=1)
distribution = np.bincount(y)
print(f"Class distribution: {distribution}")
total = sum(distribution)
percentage = [round(distribution[0]/total, 5), round(distribution[1]/total, 5), round(distribution[2]/total, 5)]
print(f"Class distribution percent: {percentage}")

Class distribution: [20064 19652 17761]
Class distribution percent: [np.float64(0.34908), np.float64(0.34191), np.float64(0.30901)]


It appears ties are the least common, but a distribution of 35%, 34%, and 31% means all possibilities are almost equally likely to ocur. 

In [26]:
# ============================================================
# 6. TRAIN WITH CROSS-VALIDATION
# ============================================================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros((len(X_all), 3))
test_preds = np.zeros((len(X_test), 3))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all, y)):
    model = XGBClassifier(
        objective='multi:softprob',
        num_class=3,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss',
        early_stopping_rounds=50,
        random_state=42,
        verbosity=0
    )
    model.fit(
        X_all[tr_idx], y[tr_idx],
        eval_set=[(X_all[va_idx], y[va_idx])],
        verbose=False
    )

    oof_preds[va_idx] = model.predict_proba(X_all[va_idx])
    test_preds += model.predict_proba(X_test) / 5

    print(f"Fold {fold+1}: log_loss = {log_loss(y[va_idx], oof_preds[va_idx]):.4f}")

print(f"\nOverall CV log_loss: {log_loss(y, oof_preds):.4f}")


c:\Users\jeffj\OneDrive\Desktop\AWS S3 files\.venv\Lib\site-packages\sklearn\metrics\_classification.py:310: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


Fold 1: log_loss = 1.0299


c:\Users\jeffj\OneDrive\Desktop\AWS S3 files\.venv\Lib\site-packages\sklearn\metrics\_classification.py:310: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


Fold 2: log_loss = 1.0310


c:\Users\jeffj\OneDrive\Desktop\AWS S3 files\.venv\Lib\site-packages\sklearn\metrics\_classification.py:310: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


Fold 3: log_loss = 1.0262


c:\Users\jeffj\OneDrive\Desktop\AWS S3 files\.venv\Lib\site-packages\sklearn\metrics\_classification.py:310: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


Fold 4: log_loss = 1.0274
Fold 5: log_loss = 1.0293

Overall CV log_loss: 1.0288


c:\Users\jeffj\OneDrive\Desktop\AWS S3 files\.venv\Lib\site-packages\sklearn\metrics\_classification.py:310: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
c:\Users\jeffj\OneDrive\Desktop\AWS S3 files\.venv\Lib\site-packages\sklearn\metrics\_classification.py:310: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


with a overall CV log_loss: 1.0326, this is better than a purely random guess, which has a log loss of around 1.09, but it could be better. Lets try adding a feature that checks for similar values. We run the following code again with our new feature below: 

In [25]:
# ============================================================
# 4b. SIMILARITY FEATURES
# ============================================================

def word_overlap(a, b):
    words_a = set(a.lower().split())
    words_b = set(b.lower().split())
    if not words_a or not words_b:
        return 0.0
    intersection = words_a & words_b
    return len(intersection) / min(len(words_a), len(words_b))

# Word overlap — what fraction of words are shared
train_features['word_overlap'] = train.apply(
    lambda row: word_overlap(row['response_a'], row['response_b']), axis=1
)
test_features['word_overlap'] = test.apply(
    lambda row: word_overlap(row['response_a'], row['response_b']), axis=1
)

from numpy.linalg import norm

# cosine_sim = (A · B) / (|A| × |B|)
def cosine_sim(a, b):
    dot = np.sum(a * b, axis=1)
    norm_a = norm(a, axis=1) + 1e-8
    norm_b = norm(b, axis=1) + 1e-8
    return dot / (norm_a * norm_b)

# Cosine similarity from TF-IDF vectors (already computed above)
train_features['tfidf_cosine_sim'] = cosine_sim(train_tfidf_a, train_tfidf_b)
test_features['tfidf_cosine_sim'] = cosine_sim(test_tfidf_a, test_tfidf_b)

# Combine all features
X_all = np.hstack([train_features.values, train_tfidf_prompt, train_tfidf_diff])
X_test = np.hstack([test_features.values, test_tfidf_prompt, test_tfidf_diff])
print(f"Total features: {X_all.shape[1]}")

Total features: 416
